In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("outputs/player_total_value_master_enriched_final.csv")


In [2]:
player_agg = df.groupby("player").agg(
    career_bat_rpa=("bat_rpa_raw", "sum"),
    career_bowl_rpa=("bowl_rpa_raw", "sum"),
    career_bat_wpa=("bat_wpa_raw", "sum"),
    career_bowl_wpa=("bowl_wpa_raw", "sum"),

    balls_faced_rpa=("balls_faced_rpa", "sum"),
    balls_faced_wpa=("balls_faced_wpa", "sum"),
    balls_bowled=("balls_bowled", "sum"),

    matches_played=("last_match_played", "count"),
    last_match_played=("last_match_played", "max")
).reset_index()

player_agg["balls_faced"] = (
    player_agg["balls_faced_rpa"] + player_agg["balls_faced_wpa"]
)


In [3]:
def infer_role(row):
    bf = row["balls_faced"]
    bb = row["balls_bowled"]

    if bf >= 300 and bf >= 3 * bb:
        return "batter"
    if bb >= 300 and bb >= 3 * bf:
        return "bowler"
    if bf >= 200 and bb >= 200:
        return "allrounder"
    return "fringe"

player_agg["role"] = player_agg.apply(infer_role, axis=1)


In [4]:
for col in [
    "career_bat_rpa",
    "career_bowl_rpa",
    "career_bat_wpa",
    "career_bowl_wpa"
]:
    mu = player_agg[col].mean()
    sigma = player_agg[col].std(ddof=0)
    player_agg[col + "_z"] = (player_agg[col] - mu) / sigma


In [5]:
player_agg["total_value_z"] = (
    player_agg["career_bat_rpa_z"] +
    player_agg["career_bowl_rpa_z"] +
    player_agg["career_bat_wpa_z"] +
    player_agg["career_bowl_wpa_z"]
)


In [6]:
player_agg["last_match_played"] = pd.to_datetime(
    player_agg["last_match_played"]
)

today = player_agg["last_match_played"].max()
player_agg["days_since_last_match"] = (
    today - player_agg["last_match_played"]
).dt.days

player_agg["active_12m"] = player_agg["days_since_last_match"] <= 365
player_agg["active_24m"] = player_agg["days_since_last_match"] <= 730
player_agg["active_36m"] = player_agg["days_since_last_match"] <= 1095


In [7]:
final_cols = [
    "player",

    "career_bat_rpa",
    "career_bowl_rpa",
    "career_bat_wpa",
    "career_bowl_wpa",

    "career_bat_rpa_z",
    "career_bowl_rpa_z",
    "career_bat_wpa_z",
    "career_bowl_wpa_z",

    "total_value_z",

    "balls_faced",
    "balls_bowled",
    "matches_played",

    "role",

    "last_match_played",
    "days_since_last_match",
    "active_12m",
    "active_24m",
    "active_36m"
]

player_vectors = player_agg[final_cols]


In [8]:
player_vectors.to_csv(
    "outputs/player_vectors_base.csv",
    index=False
)


In [9]:
player_vectors["role"].value_counts()

role
fringe        2414
batter         583
bowler         387
allrounder     174
Name: count, dtype: int64

In [10]:
player_vectors.sort_values(
    "total_value_z", ascending=False
).head(10)

,player,career_bat_rpa,career_bowl_rpa,career_bat_wpa,career_bowl_wpa,career_bat_rpa_z,career_bowl_rpa_z,career_bat_wpa_z,career_bowl_wpa_z,total_value_z,balls_faced,balls_bowled,matches_played,role,last_match_played,days_since_last_match,active_12m,active_24m,active_36m
2892,SP Narine,330.255438,1201.591867,-0.934508,11.405882,5.144345,23.128155,-1.540008,20.473329,47.205821,2929.0,4499.0,1,allrounder,2025-07-05,21,True,True,True
1018,GJ Maxwell,926.283009,-61.816390,7.087060,-1.010478,14.687618,-1.009765,10.707008,-1.830101,22.554760,5993.0,1446.0,1,batter,2025-07-26,0,True,True,True
2672,Rashid Khan,105.502166,481.206109,-1.907995,7.776876,1.545716,9.364897,-3.026290,13.954568,21.838891,1147.0,3759.0,1,bowler,2025-05-30,57,True,True,True
2170,N Pooran,747.722923,0.000000,6.116756,0.000000,11.828610,0.171262,9.225587,-0.014986,21.210472,6513.0,0.0,1,batter,2025-07-13,13,True,True,True
78,AD Hales,254.911179,0.000000,11.022656,0.000000,3.937973,0.171262,16.715722,-0.014986,20.809971,7838.0,0.0,1,batter,2025-07-06,20,True,True,True
621,CA Lynn,527.609178,0.000000,6.971165,0.000000,8.304267,0.171262,10.530065,-0.014986,18.990607,5003.0,0.0,1,batter,2025-07-05,21,True,True,True
1411,JM Vince,377.263937,-1.158065,7.749317,0.000000,5.897020,0.149137,11.718116,-0.014986,17.749286,8285.0,0.0,1,batter,2025-07-13,13,True,True,True
1364,JJ Bumrah,-6.665292,471.796587,0.000000,4.007605,-0.250249,9.185124,-0.113238,7.183851,16.005489,31.0,2220.0,1,bowler,2025-06-01,55,True,True,True
1317,JC Buttler,508.254709,0.000000,4.938766,0.000000,7.994374,0.171262,7.427078,-0.014986,15.577727,7375.0,0.0,1,batter,2025-07-18,8,True,True,True
767,DA Warner,475.454215,0.000000,5.202857,0.000000,7.469190,0.171262,7.830281,-0.014986,15.455746,6146.0,0.0,1,batter,2025-07-05,21,True,True,True


In [11]:
player_vectors[player_vectors["active_12m"] == False].shape


(2339, 19)